## Bibliotecas

In [1]:
from ultralytics import YOLO
import ultralytics
import os
import random
import shutil
import yaml

ultralytics.checks()

Ultralytics 8.4.33 🚀 Python-3.10.20 torch-2.11.0 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Setup complete ✅ (22 CPUs, 15.3 GB RAM, 117.0/1006.9 GB disk)


## Split do Dataset

In [2]:
def build_physical_subset(
    original_yaml_path: str,
    output_dir: str,
    n_train: int = 1000,
    n_val: int = 150,
    n_test: int = 300,
    seed: int = 42
) -> str:
    random.seed(seed)

    with open(original_yaml_path) as f:
        cfg = yaml.safe_load(f)

    base_path = cfg['path']
    splits = {'train': n_train, 'val': n_val, 'test': n_test}
    
    new_dataset_dir = os.path.join(output_dir, 'dataset_subset')
    os.makedirs(new_dataset_dir, exist_ok=True)
    
    valid_extensions = ('.jpg', '.jpeg', '.png')

    for split, n in splits.items():
        img_dir = os.path.join(base_path, 'images', split)
        all_files = os.listdir(img_dir)
        
        all_images = [f for f in all_files if f.lower().endswith(valid_extensions)]
        
        valid_pairs = []
        for img_name in all_images:
            img_path = os.path.join(img_dir, img_name)
            
            label_name = img_name.rsplit('.', 1)[0] + '.txt'
            label_path = os.path.join(base_path, 'labels', split, label_name)
            
            if os.path.exists(label_path):
                valid_pairs.append((img_path, label_path))

        if len(valid_pairs) < n:
            raise ValueError(f"[{split}] Apenas {len(valid_pairs)} pares válidos encontrados. Precisava de {n}.")

        selected_pairs = random.sample(valid_pairs, n)

        dest_img_dir = os.path.join(new_dataset_dir, 'images', split)
        dest_lbl_dir = os.path.join(new_dataset_dir, 'labels', split)
        os.makedirs(dest_img_dir, exist_ok=True)
        os.makedirs(dest_lbl_dir, exist_ok=True)

        for img_src, lbl_src in selected_pairs:
            shutil.copy2(img_src, dest_img_dir)
            shutil.copy2(lbl_src, dest_lbl_dir)

        print(f"[{split}] {n} imagens e anotações copiadas para -> {dest_img_dir}")

    new_cfg = {
        'path': new_dataset_dir,
        'train': 'images/train',
        'val':   'images/val',
        'test':  'images/test',
        'names': cfg['names']
    }

    new_yaml_path = os.path.join(output_dir, 'dangerousItems_subset.yaml')
    with open(new_yaml_path, 'w') as f:
        yaml.dump(new_cfg, f, default_flow_style=False, allow_unicode=True)

    print(f"\nYML gerado: {new_yaml_path}")
    return new_yaml_path

## YOLO26

In [3]:
class Yolo26():

    def __init__(self):
        pass

    def set_model(self, model_name: str):
        return YOLO(model_name)

    def train(self, model, save_path: str, yaml_path: str,
              epochs: int = 100, patience: int = 20,
              freeze: int = 10, lr: float = 1e-3, save: bool = True):
        """Treina o modelo com base no yaml"""
        return model.train(
            data=yaml_path,
            epochs=epochs,
            patience=patience,
            freeze=freeze,
            lr0=lr,
            save=save,
            project=save_path,
            optimizer='Adam',
            # os que tão aq foram meus testes para melhorar um pouco o desempenho
            imgsz=640,
            batch=32,
            cos_lr=True,
            close_mosaic=15,
            cache='disk',
            #degrees=45.0,
            #fliplr=0.5,
        )

    def get_best_model_path(self, train_results) -> str:
        """Retorna o caminho do melhor modelo salvo após o treino"""
        return os.path.join(str(train_results.save_dir), 'weights', 'best.pt')

    def val(self, best_model_path: str):
        """Valida o melhor modelo treinado no conjunto de validação"""
        model = YOLO(best_model_path)
        return model.val()
    
    def test(self, best_model_path: str, yaml_path: str):
        """Avalia o melhor modelo treinado no conjunto de teste"""
        model = YOLO(best_model_path)
        return model.val(data=yaml_path, split='test')

    def predict(self, best_model_path: str, test_path: str):
        """Realiza predição com o melhor modelo treinado"""
        model = YOLO(best_model_path)
        return model(test_path)

## Setup

In [4]:
root = os.getcwd()
results_dir = os.path.join(root, 'resultados')
os.makedirs(results_dir, exist_ok=True)

model_paths = {
    'nano': {
        'train': os.path.join(results_dir, 'train', 'nano'),
        'val': os.path.join(results_dir, 'val',   'nano'),
        'test': os.path.join(results_dir, 'test',  'nano'),
    },
    'small': {
        'train': os.path.join(results_dir, 'train', 'small'),
        'val': os.path.join(results_dir, 'val',   'small'),
        'test': os.path.join(results_dir, 'test',  'small'),
    },
}

subset_yaml = build_physical_subset(
    original_yaml_path='dangerousItems.yaml',
    output_dir=results_dir,
    n_train=1000,
    n_val=150,
    n_test=300,
    seed=42
)

yolo26 = Yolo26()

[train] 1000 imagens e anotações copiadas para -> /home/henry/yolo_dangerous_items/resultados/dataset_subset/images/train
[val] 150 imagens e anotações copiadas para -> /home/henry/yolo_dangerous_items/resultados/dataset_subset/images/val
[test] 300 imagens e anotações copiadas para -> /home/henry/yolo_dangerous_items/resultados/dataset_subset/images/test

YML gerado: /home/henry/yolo_dangerous_items/resultados/dangerousItems_subset.yaml


## Nano

### Treinamento

In [5]:
model_nano = yolo26.set_model('yolo26n.pt')

results_train_nano = yolo26.train(
    model_nano,
    save_path=model_paths['nano']['train'],
    yaml_path=subset_yaml
)

best_nano = yolo26.get_best_model_path(results_train_nano)
print(f"Melhor modelo nano: {best_nano}")

New https://pypi.org/project/ultralytics/8.4.60 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.33 🚀 Python-3.10.20 torch-2.11.0 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=15, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/henry/yolo_dangerous_items/resultados/dangerousItems_subset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momen

### Validação

In [8]:
metrics_nano = yolo26.val(best_nano)

print(f"\n[Nano] mAP@50: {metrics_nano.box.map50:.4f}")
print(f"[Nano] mAP@50-95: {metrics_nano.box.map:.4f}")
print(f"[Nano] Precision: {metrics_nano.box.mp:.4f}")
print(f"[Nano] Recall: {metrics_nano.box.mr:.4f}")

Ultralytics 8.4.33 🚀 Python-3.10.20 torch-2.11.0 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO26n summary (fused): 122 layers, 2,375,811 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 282.1±64.4 MB/s, size: 254.0 KB)
val: Scanning /home/henry/yolo_dangerous_items/resultados/dataset_subset/labels/val.cache... 150 images, 14 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 150/150 39.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 6.5it/s 1.5s.1ss
                   all        150        158      0.713      0.696      0.739      0.484
               machete         32         33      0.627      0.667      0.623      0.449
                 knife         27         27      0.744      0.538      0.641      0.426
          baseball bat         32         32      0.725      0.844      0.825      0.551
                 rifle         30         31      0.787      0.774      

### Teste

In [9]:
metrics_test_nano = yolo26.test(
    best_model_path=best_nano, 
    yaml_path=subset_yaml
)

print(f"\n[Nano TESTE] mAP@50: {metrics_test_nano.box.map50:.4f}")
print(f"[Nano TESTE] mAP@50-95: {metrics_test_nano.box.map:.4f}")

Ultralytics 8.4.33 🚀 Python-3.10.20 torch-2.11.0 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO26n summary (fused): 122 layers, 2,375,811 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4015.0±1598.1 MB/s, size: 220.8 KB)
val: Scanning /home/henry/yolo_dangerous_items/resultados/dataset_subset/labels/test.cache... 300 images, 26 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 300/300 83.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 8.6it/s 2.2s<0.1s
                   all        300        313      0.762       0.57      0.683      0.435
               machete         69         75      0.715      0.536      0.641      0.458
                 knife         48         49      0.663      0.408      0.562      0.336
          baseball bat         58         58      0.848      0.671      0.747      0.542
                 rifle         67         73      0.774      0.685 

### Referências Úteis

[Docs Train Ultralytics](https://docs.ultralytics.com/modes/train#augmentation-settings-and-hyperparameters)